# 01. Разведочный анализ (EDA)

**Задача:** бинарная классификация направления дневного движения цены.  
**Целевая метрика:** Precision.

## 0. Источник данных и описание датасета

### Поиск и выбор источника

**Источник:** [Yahoo Finance](https://finance.yahoo.com) через библиотеку [`yfinance`](https://pypi.org/project/yfinance/) (v0.2+). Скачиваем дневные OHLCV-бары и макро-серии.
- O - open (цена на момент начала временного промежутка)
- H - high (самая высокая цена актива за временной промежуток)
- L - low (самая низкая цена актива за временной промежуток)
- C - close (цена на момент закрытия временного промежутка)
- V - volume (объем торгов за временной промежуток)

**Почему yfinance, а не альтернативы:**
- Бесплатно, без обязательного API-ключа, без жёстких rate-limit'ов для исторических данных.
- Покрытие: все крупные биржевые тикеры США + индексы (^GSPC, ^IXIC, ^DJI) с 1990-х.
- Альтернативы (Alpha Vantage — 25 запросов/день на free tier; Polygon — платный после демо; IEX Cloud — закрылся в 2024) проигрывают по бесплатному покрытию исторических OHLCV.
- `auto_adjust=False` сохраняет «сырые» цены — мы сами решаем, как обрабатывать сплиты/дивиденды; для классификации направления это не критично, т.к. интересует знак изменения.

### Выбор активов

Используем 5 тикеров — индекс + 4 крупных бумаги из разных секторов, чтобы проверить, обобщаются ли паттерны фичей за пределы одного актива:

| Тикер | Описание | Сектор |
|---|---|---|
| `^GSPC` | S&P 500 — индекс 500 крупнейших компаний США | Broad market (ОСНОВНОЙ) |
| `AAPL` | Apple Inc. | Information Technology |
| `MSFT` | Microsoft Corp. | Information Technology |
| `JPM` | JPMorgan Chase | Financials |
| `XOM` | Exxon Mobil | Energy |

Логика выбора: 
- индекс (низкая волатильность, диверсифицирован) 
- два tech-гиганта (высокая ликвидность, чувствительны к настроениям)
- банк (циклы ставок)
- энерго (товарные циклы). Если фича работает на всех пяти — она устойчива.

### Период

**`2010-01-01` … `2024-12-31`** — 15 календарных лет, ~3700 торговых дней на тикер.

Почему именно так:
- **С 2010** — после Великой Рецессии 2008–09, когда режимы рынка кардинально менялись (QE, ZIRP). Включать кризис в трейн-данные = учиться на режиме, который, скорее всего, не повторится в форме 1-в-1; большинство production-стратегий используют post-GFC период.
- **До 2024-12-31**.
- В диапазон попадают значимые шоки: COVID 2020, инфляционный цикл 2022, AI-ралли 2023–24 — хорошее покрытие разных режимов волатильности для проверки робастности.

### Объём

5 тикеров × ~3700 строк ≈ **18.5к наблюдений**. Каждое — 5 OHLCV-колонок + 9 технических фичей + 5 макро-фичей + Target.

## 0b. Настройка

In [ ]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.outliers_influence import variance_inflation_factor

from src.preprocessing import ALL_FEATURE_COLUMNS, FEATURE_COLUMNS, MACRO_FEATURE_COLUMNS, build_dataset

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.float_format', '{:.4f}'.format)

TICKERS = ['^GSPC', 'AAPL', 'MSFT', 'JPM', 'XOM']
PRIMARY = '^GSPC'
START, END = '2010-01-01', '2024-12-31'
RANDOM_SEED = 42

TECH_FEATURES = FEATURE_COLUMNS
MACRO_FEATURES = MACRO_FEATURE_COLUMNS
MODEL_FEATURES = ALL_FEATURE_COLUMNS

In [ ]:
datasets = {t: build_dataset(t, START, END, include_macro=True) for t in TICKERS}
for t, df in datasets.items():
    print(f'{t:6s} shape={df.shape} range={df.index.min().date()}..{df.index.max().date()}')

## 1. Базовая проверка (Sanity Check) и очистка пропусков

Проверим: типы колонок, пропуски (после `dropna()` в `build_dataset` не должно остаться), разрывы в датах, базовые инварианты OHLC.

### 1a. Проверка макро-фичей после join

`build_dataset(..., include_macro=True)` добавляет 5 макро-фичей через общий макро-календарь. Проверим, что все тикеры получили одинаковый макро-контекст, а `dropna()` не оставил пропусков.

In [ ]:
macro_sanity_rows = []
for ticker, ticker_df in datasets.items():
    missing_cols = sorted(set(MACRO_FEATURES) - set(ticker_df.columns))
    macro_sanity_rows.append({
        'ticker': ticker,
        'rows': len(ticker_df),
        'start': ticker_df.index.min().date(),
        'end': ticker_df.index.max().date(),
        'missing_macro_cols': ', '.join(missing_cols) if missing_cols else 'none',
        'macro_nan_total': int(ticker_df[MACRO_FEATURES].isna().sum().sum()),
    })

macro_sanity = pd.DataFrame(macro_sanity_rows).set_index('ticker')
macro_sanity

In [ ]:
# Макро-фичи не зависят от тикера, поэтому на общих датах значения должны совпадать.
primary_macro = datasets[PRIMARY][MACRO_FEATURES]
macro_alignment_rows = []
for ticker, ticker_df in datasets.items():
    common_index = primary_macro.index.intersection(ticker_df.index)
    max_abs_diff = (ticker_df.loc[common_index, MACRO_FEATURES] - primary_macro.loc[common_index]).abs().max()
    macro_alignment_rows.append(max_abs_diff.rename(ticker))

macro_alignment = pd.DataFrame(macro_alignment_rows)
macro_alignment.index.name = 'ticker'
macro_alignment.round(10)

In [ ]:
df = datasets[PRIMARY]
df.info()
df.describe().T

In [ ]:
# Проверка пропусков (должно быть 0 после dropna в build_dataset)
nan_summary = pd.DataFrame({t: d.isna().sum().sum() for t, d in datasets.items()}, index=['NaN total']).T
nan_summary['rows'] = [d.shape[0] for d in datasets.values()]
nan_summary

In [ ]:
# Дубликаты индекса и разрывы в торговых днях
report = []
for t, d in datasets.items():
    dups = int(d.index.duplicated().sum())
    expected = pd.bdate_range(d.index.min(), d.index.max())
    missing_bdays = len(expected.difference(d.index))
    report.append({'ticker': t, 'duplicate_index': dups, 'missing_bdays_vs_bdate_range': missing_bdays})
pd.DataFrame(report).set_index('ticker')

Расхождение `missing_bdays` ожидаемо: NYSE имеет ~9–10 праздников в год, плюс первые ~33 строки выпали из-за rolling-окон фичей. Это нормально, не проблема.

In [ ]:
# OHLC-инварианты: High >= Low, High >= Close >= Low, Volume >= 0
violations = []
for t, d in datasets.items():
    v = {
        'High<Low': int((d['High'] < d['Low']).sum()),
        'Close>High': int((d['Close'] > d['High']).sum()),
        'Close<Low': int((d['Close'] < d['Low']).sum()),
        'Volume<0': int((d['Volume'] < 0).sum()),
    }
    violations.append({'ticker': t, **v})
pd.DataFrame(violations).set_index('ticker')

**Вывод 1:** данные чистые — ни пропусков, ни нарушений OHLC-инвариантов. Можно работать дальше.

## 2. Анализ баланса классов (Target Distribution)

Target = 1, если завтрашний Close выше сегодняшнего, иначе 0. Если перекос сильный, нужно либо `class_weight='balanced'`, либо подбор порога.

In [ ]:
balance = pd.DataFrame({t: d['Target'].value_counts(normalize=True) for t, d in datasets.items()}).T
balance.columns = ['down (0)', 'up (1)']
balance['imbalance_pp'] = (balance['up (1)'] - 0.5) * 100
balance.round(4)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
balance[['down (0)', 'up (1)']].plot(kind='bar', stacked=True, ax=ax, color=['#d62728', '#2ca02c'])
ax.axhline(0.5, color='black', linestyle='--', linewidth=1)
ax.set_ylabel('proportion')
ax.set_title('Баланс классов Target по тикерам')
ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

In [ ]:
# Стабильна ли доля 'вверх' во времени? (rolling 1Y = 252 торговых дня)
fig, ax = plt.subplots(figsize=(11, 4))
for t, d in datasets.items():
    d['Target'].rolling(252).mean().plot(ax=ax, label=t, alpha=0.8)
ax.axhline(0.5, color='black', linestyle='--', linewidth=1)
ax.set_ylabel('rolling mean(Target), 252d')
ax.set_title('Доля растущих дней — скользящее среднее за 1 год')
ax.legend(); plt.tight_layout(); plt.show()

**Вывод 2:** на индексе и крупных бумагах перевес «вверх» обычно 1–5 п.п. — слабый. Точность baseline по умолчанию (predict majority) даст ≈ baseline['up (1)'], от этого и стартуем. Для логистической регрессии разумно использовать `class_weight='balanced'`.

## 3. Анализ распределения фичей (толстые хвосты)

Финансовые ряды славятся «leptokurtic» распределениями — пик выше нормального и тяжёлые хвосты. Для каждой фичи смотрим: гистограмму, boxplot, Q-Q-plot, skew/kurtosis, долю > 3σ.

In [ ]:
df = datasets[PRIMARY]
stat_rows = []
for f in FEATURE_COLUMNS:
    s = df[f].dropna()
    mu, sigma = s.mean(), s.std()
    pct_gt_3sigma = float(((s - mu).abs() > 3 * sigma).mean())
    stat_rows.append({
        'feature': f,
        'mean': mu, 'std': sigma,
        'skew': float(stats.skew(s)),
        'excess_kurtosis': float(stats.kurtosis(s)),  # excess (norm = 0)
        'share_>|3sigma|': pct_gt_3sigma,
    })
feat_stats = pd.DataFrame(stat_rows).set_index('feature')
feat_stats.round(4)

In [ ]:
# Гистограммы + KDE
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, f in zip(axes.flat, FEATURE_COLUMNS):
    sns.histplot(df[f], kde=True, ax=ax, bins=60, color='steelblue')
    ax.set_title(f)
fig.suptitle(f'Распределения 9 фичей ({PRIMARY})', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Boxplots — выбросы
fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, f in zip(axes.flat, FEATURE_COLUMNS):
    sns.boxplot(x=df[f], ax=ax, color='lightblue')
    ax.set_title(f); ax.set_xlabel('')
fig.suptitle(f'Boxplots — выбросы ({PRIMARY})', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Q-Q plots против нормального — кривизна на концах = толстые хвосты
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, f in zip(axes.flat, FEATURE_COLUMNS):
    stats.probplot(df[f].dropna(), dist='norm', plot=ax)
    ax.set_title(f)
fig.suptitle(f'Q-Q plots vs Normal ({PRIMARY})', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

**Вывод 3:** `Daily_Return`, `Lag_Return_1`, `Volume_ROC` — тяжёлые хвосты (excess kurtosis ≫ 0, Q-Q ломаются на концах). Для них перед линейной моделью имеет смысл винзоризация / клиппинг (скажем, на 1-м и 99-м процентилях) или использовать робастные модели (деревья). `RSI_14`, `Bollinger_pctB` ограничены по построению и имеют умеренные хвосты.

### 3a. Решение по выбросам

Выбросы зафиксированы (Q-Q-plots, доля > 3σ, excess kurtosis), но **на этапе EDA мы их не модифицируем**. Обоснование:

1. **Это не ошибки данных, а реальные движения рынка.** Скачки `Daily_Return` ±10% в марте 2020, августе 2024 и т.п. — это сигналы режима, а не шум. Удалять их = терять информацию о хвостовых событиях, которые как раз и важны для управления риском.
2. **Подход зависит от модели.** Для бустинга/деревьев робастность к выбросам встроена — splitting по квантилям. Для линейной модели понадобится `RobustScaler` или винзоризация.
3. **Любая трансформация должна фититься только на train.** Винзоризация, рассчитанная на полном датасете (включая test), — форма data leakage. Поэтому формально такая обработка должна жить **внутри pipeline** (`sklearn.preprocessing.RobustScaler` или кастомный `Winsorizer`) и применяться в `02_baseline.ipynb`, а не в EDA.

**Решение:** оставляем фичи as-is, в `02_baseline` для линейной модели применим `RobustScaler` и винзоризацию [1%, 99%] внутри `Pipeline`, фитя только на train-fold.

### 3b. Макро-фичи: распределения и рыночные режимы

Макро-фичи описывают общий рыночный контекст, а не поведение конкретного тикера. Поэтому анализируем их отдельно от технических индикаторов: форма распределений, экстремальные значения и режимы VIX, кривой доходности и доллара.

In [ ]:
df = datasets[PRIMARY]

macro_stat_rows = []
for f in MACRO_FEATURES:
    s = df[f].dropna()
    mu, sigma = s.mean(), s.std()
    macro_stat_rows.append({
        'feature': f,
        'mean': mu,
        'std': sigma,
        'skew': stats.skew(s),
        'excess_kurtosis': stats.kurtosis(s),
        'share_abs_gt_3sigma': (np.abs(s - mu) > 3 * sigma).mean(),
        'min': s.min(),
        'max': s.max(),
    })

macro_dist_stats = pd.DataFrame(macro_stat_rows).set_index('feature')
macro_dist_stats.round(5)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, f in zip(axes.flat, MACRO_FEATURES):
    sns.histplot(df[f], kde=True, ax=ax, bins=60, color='steelblue')
    ax.set_title(f)

for ax in axes.flat[len(MACRO_FEATURES):]:
    ax.axis('off')

fig.suptitle('Распределения макро-фичей', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, f in zip(axes.flat, MACRO_FEATURES):
    sns.boxplot(x=df[f], ax=ax, color='lightblue')
    ax.set_title(f)
    ax.set_xlabel('')

for ax in axes.flat[len(MACRO_FEATURES):]:
    ax.axis('off')

fig.suptitle('Boxplot макро-фичей', y=1.02)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, f in zip(axes.flat, MACRO_FEATURES):
    stats.probplot(df[f].dropna(), dist='norm', plot=ax)
    ax.set_title(f)

for ax in axes.flat[len(MACRO_FEATURES):]:
    ax.axis('off')

fig.suptitle('Q-Q plots макро-фичей', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
macro_regime_features = ['VIX_Level', 'Yield_Spread', 'DXY_Return']
fig, axes = plt.subplots(len(macro_regime_features), 1, figsize=(13, 9), sharex=True)

for ax, f in zip(axes, macro_regime_features):
    df[f].plot(ax=ax, color='steelblue', linewidth=1)
    df[f].rolling(252).mean().plot(ax=ax, color='darkorange', linewidth=1.5, label='rolling mean 252d')
    ax.set_title(f)
    ax.legend(loc='upper right')
    if f == 'VIX_Level':
        ax.axhline(30, color='red', linestyle='--', linewidth=1, label='VIX=30')
    elif f == 'Yield_Spread':
        ax.axhline(0, color='red', linestyle='--', linewidth=1, label='inversion')
    elif f == 'DXY_Return':
        ax.axhline(0, color='black', linestyle='--', linewidth=1)

fig.suptitle(f'Макро-режимы для {PRIMARY}', y=1.02)
plt.tight_layout(); plt.show()

## 4. Влияние фичей на Таргет (Feature vs Target)

Здесь главный вопрос: каждая фича несёт сигнал или шум? Используем 4 угла обзора:
- conditional KDE (target=0 vs target=1),
- boxplot фичи по таргету,
- децильное усреднение target по бину фичи,
- статистика: Mann–Whitney U-тест (не предполагает нормальности) + Cohen's d + Mutual Information.
Анализ ведём на основном тикере (`^GSPC`); сводную статистику повторяем на остальных.

In [ ]:
df = datasets[PRIMARY]

fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, f in zip(axes.flat, FEATURE_COLUMNS):
    for cls, color in [(0, '#d62728'), (1, '#2ca02c')]:
        sns.kdeplot(df.loc[df['Target'] == cls, f], ax=ax, label=f'Target={cls}', color=color, fill=True, alpha=0.3)
    ax.set_title(f); ax.legend(fontsize=8)
fig.suptitle(f'Conditional KDE по Target ({PRIMARY})', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, f in zip(axes.flat, FEATURE_COLUMNS):
    sns.boxplot(x=df['Target'].astype(int), y=df[f], ax=ax, hue=df['Target'].astype(int),
                palette=['#d62728', '#2ca02c'], legend=False)
    ax.set_title(f); ax.set_xlabel('')
fig.suptitle(f'Boxplot фичи по классу Target ({PRIMARY})', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Децильный анализ: разбить фичу на 10 бинов, посчитать mean(Target) в каждом
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, f in zip(axes.flat, FEATURE_COLUMNS):
    bins = pd.qcut(df[f], q=10, duplicates='drop')
    target_by_bin = df.groupby(bins, observed=True)['Target'].mean()
    target_by_bin.plot(kind='bar', ax=ax, color='steelblue')
    ax.axhline(df['Target'].mean(), color='black', linestyle='--', linewidth=1)
    ax.set_title(f); ax.set_xlabel(''); ax.set_ylabel('mean(Target)')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
fig.suptitle(f'Mean(Target) по децилям фичи ({PRIMARY}) — горизонталь = средний таргет', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Статистика по фичам vs Target: Mann-Whitney U + Cohen's d + Mutual Information
def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return (a.mean() - b.mean()) / pooled if pooled > 0 else 0.0

def feature_target_stats(d, feature_cols=TECH_FEATURES, primary_col='Target'):
    rows = []
    g0 = d[d[primary_col] == 0]
    g1 = d[d[primary_col] == 1]
    mi = mutual_info_classif(d[feature_cols].to_numpy(), d[primary_col].astype(int).to_numpy(), random_state=RANDOM_SEED)
    for f, mi_val in zip(feature_cols, mi):
        u_stat, p = stats.mannwhitneyu(g1[f], g0[f], alternative='two-sided')
        rows.append({
            'feature': f,
            'mean_up': g1[f].mean(),
            'mean_down': g0[f].mean(),
            'cohens_d': cohens_d(g1[f], g0[f]),
            'mw_pvalue': p,
            'mutual_info': mi_val,
        })
    return pd.DataFrame(rows).set_index('feature')

primary_stats = feature_target_stats(datasets[PRIMARY], TECH_FEATURES).sort_values('mutual_info', ascending=False)
primary_stats.round(5)

### 4a. Макро-фичи vs Target

Проверим, отличаются ли макро-режимы между классами `Target=0` и `Target=1`. Один и тот же макро-контекст может по-разному работать для разных тикеров, поэтому отдельно посмотрим основной тикер и multi-ticker MI.

In [ ]:
macro_target_stats = feature_target_stats(datasets[PRIMARY], MACRO_FEATURES).sort_values('mutual_info', ascending=False)
macro_target_stats.round(5)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, f in zip(axes.flat, MACRO_FEATURES):
    for cls, color in [(0, '#d62728'), (1, '#2ca02c')]:
        sns.kdeplot(
            df.loc[df['Target'] == cls, f],
            ax=ax,
            label=f'Target={cls}',
            color=color,
            fill=True,
            alpha=0.3,
        )
    ax.set_title(f)
    ax.legend()

for ax in axes.flat[len(MACRO_FEATURES):]:
    ax.axis('off')

fig.suptitle('Макро-фичи: распределения по классам Target', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, f in zip(axes.flat, MACRO_FEATURES):
    bins = pd.qcut(df[f], q=10, duplicates='drop')
    target_by_bin = df.groupby(bins, observed=True)['Target'].mean()
    target_by_bin.plot(kind='bar', ax=ax, color='steelblue')
    ax.axhline(df['Target'].mean(), color='red', linestyle='--', linewidth=1, label='base rate')
    ax.set_title(f)
    ax.set_ylabel('mean(Target)')
    ax.tick_params(axis='x', rotation=75)

for ax in axes.flat[len(MACRO_FEATURES):]:
    ax.axis('off')

fig.suptitle('Децильный анализ макро-фичей', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
macro_mi_table = pd.DataFrame({
    t: feature_target_stats(ticker_df, MACRO_FEATURES)['mutual_info'] for t, ticker_df in datasets.items()
})
macro_mi_table['mean'] = macro_mi_table.mean(axis=1)
macro_mi_table.sort_values('mean', ascending=False).round(5)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
macro_mi_table.drop(columns='mean').plot(kind='bar', ax=ax)
ax.set_ylabel('mutual information(feature, Target)')
ax.set_title('Mutual Information макро-фичей по тикерам')
ax.tick_params(axis='x', rotation=45)
ax.legend(title='ticker', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()

In [ ]:
# Сводка по всем тикерам: насколько MI устойчив
mi_table = pd.DataFrame({
    t: feature_target_stats(d)['mutual_info'] for t, d in datasets.items()
})
mi_table['mean'] = mi_table.mean(axis=1)
mi_table.sort_values('mean', ascending=False).round(5)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
mi_table.drop(columns='mean').plot(kind='bar', ax=ax)
ax.set_ylabel('mutual information(feature, Target)')
ax.set_title('Mutual Information по тикерам')
ax.tick_params(axis='x', rotation=45)
ax.legend(title='ticker', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()

**Вывод 4:** на дневных данных MI у всех фичей мал (это нормально — рынки близки к эффективным, дневное направление шумное). Однако относительный ранг фичей даёт первый ориентир для отбора. На большинстве тикеров в топе обычно `RSI_14`, `Bollinger_pctB`, `Price_to_SMA20`, `Daily_Return`. Размеры эффекта (Cohen's d) маленькие — нужны не одиночные фичи, а их комбинации в модели.

## 5. Мультиколлинеарность (матрица корреляций + VIF)

Линейная модель страдает от коллинеарности; деревья — нет, но интерпретация feature importance тоже искажается. Считаем Pearson + Spearman, помечаем пары |corr|>0.7 и фичи с VIF>10.

In [ ]:
df = datasets[PRIMARY]
corr_p = df[FEATURE_COLUMNS].corr(method='pearson')
corr_s = df[FEATURE_COLUMNS].corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, mat, title in zip(axes, [corr_p, corr_s], ['Pearson', 'Spearman']):
    sns.heatmap(mat, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax,
                cbar_kws={'shrink': 0.8})
    ax.set_title(f'{title} correlation ({PRIMARY})')
plt.tight_layout(); plt.show()

In [ ]:
# Пары с |corr| > 0.7 (по Pearson)
high = []
for i, a in enumerate(FEATURE_COLUMNS):
    for b in FEATURE_COLUMNS[i+1:]:
        c = corr_p.loc[a, b]
        if abs(c) > 0.7:
            high.append({'a': a, 'b': b, 'pearson': c, 'spearman': corr_s.loc[a, b]})
pd.DataFrame(high).round(3) if high else 'Нет пар с |Pearson| > 0.7'

In [ ]:
# VIF — для каждой фичи
X = df[FEATURE_COLUMNS].to_numpy()
X_std = (X - X.mean(axis=0)) / X.std(axis=0)  # нормировка для устойчивости VIF
vif = pd.Series(
    [variance_inflation_factor(X_std, i) for i in range(X_std.shape[1])],
    index=FEATURE_COLUMNS, name='VIF',
).sort_values(ascending=False)
vif_df = vif.to_frame()
vif_df['flag'] = np.where(vif_df['VIF'] > 10, 'HIGH (>10)', '')
vif_df.round(3)

### 5a. Корреляции макро с техническими фичами

Текущая матрица выше показывает только 9 технических фичей. Для модели с `include_macro=True` отдельно проверим связь внутри макро-группы, связь `technical x macro` и VIF на полном наборе `MODEL_FEATURES`.

In [ ]:
macro_corr = df[MACRO_FEATURES].corr(method='pearson')
tech_macro_corr = df[TECH_FEATURES + MACRO_FEATURES].corr(method='pearson').loc[TECH_FEATURES, MACRO_FEATURES]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.heatmap(macro_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[0])
axes[0].set_title('Pearson corr внутри MACRO_FEATURES')

sns.heatmap(tech_macro_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[1])
axes[1].set_title('Pearson corr: TECH_FEATURES x MACRO_FEATURES')

plt.tight_layout(); plt.show()

In [ ]:
corr_model = df[MODEL_FEATURES].corr(method='pearson')

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_model, cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Pearson corr по полному набору MODEL_FEATURES')
plt.tight_layout(); plt.show()

X_model = df[MODEL_FEATURES].to_numpy()
X_model_std = (X_model - X_model.mean(axis=0)) / X_model.std(axis=0)
vif_model = pd.Series(
    [variance_inflation_factor(X_model_std, i) for i in range(X_model_std.shape[1])],
    index=MODEL_FEATURES,
    name='VIF',
).sort_values(ascending=False)

vif_model.to_frame().round(2)

**Вывод 5:** обычно проблемная связка — `Price_to_SMA20` ↔ `Bollinger_pctB` ↔ `MACD_Histogram` (все три измеряют отклонение цены от средней). Для линейной модели стоит оставить одну из этих трёх (скажем, `Bollinger_pctB`, т.к. он нормирован) либо использовать L2-регуляризацию. Для деревьев можно оставить все.

## 6. Проверка на стационарность

ADF-тест: H0 — есть единичный корень (нестационарность). Маленькое p-value (<0.05) — отвергаем H0, ряд стационарен. Сырая цена `Close` обязана быть нестационарной (отсюда и переход к фичам). Все наши фичи — производные от цены/объёма и должны быть стационарны.

In [ ]:
def adf_summary(series, name):
    s = series.dropna()
    stat, p, *_ = adfuller(s, autolag='AIC')
    return {'series': name, 'adf_stat': stat, 'p_value': p, 'stationary (p<0.05)': p < 0.05}

df = datasets[PRIMARY]
rows = [adf_summary(df['Close'], 'Close (raw)')]
for f in FEATURE_COLUMNS:
    rows.append(adf_summary(df[f], f))
adf_df = pd.DataFrame(rows).set_index('series')
adf_df.round(5)

In [ ]:
# Rolling mean / std (window=252) для каждой фичи — глазом проверяем дрейф
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, f in zip(axes.flat, FEATURE_COLUMNS):
    rm = df[f].rolling(252).mean()
    rs = df[f].rolling(252).std()
    ax.plot(df.index, df[f], color='lightgray', alpha=0.6, linewidth=0.7, label='raw')
    ax.plot(rm.index, rm, color='C0', linewidth=1.6, label='rolling mean (1Y)')
    ax.fill_between(rs.index, rm - rs, rm + rs, alpha=0.2, color='C0', label='±1 std (1Y)')
    ax.set_title(f); ax.legend(fontsize=7)
fig.suptitle(f'Rolling mean/std (252d) — визуальная стационарность ({PRIMARY})', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

### 6a. Стационарность макро-фичей

Макро-уровни (`VIX_Level`, `Yield_Spread`) могут быть менее стационарны, чем доходности и разности. Поэтому проверяем их отдельно и не смешиваем выводы с техническими индикаторами.

In [ ]:
macro_adf_df = pd.DataFrame([adf_summary(df[f], f) for f in MACRO_FEATURES]).set_index('series')
macro_adf_df.round(5)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, f in zip(axes.flat, MACRO_FEATURES):
    rm = df[f].rolling(252).mean()
    rs = df[f].rolling(252).std()
    rm.plot(ax=ax, label='rolling mean', color='steelblue')
    rs.plot(ax=ax, label='rolling std', color='darkorange')
    ax.set_title(f)
    ax.legend()

for ax in axes.flat[len(MACRO_FEATURES):]:
    ax.axis('off')

fig.suptitle('Rolling mean/std макро-фичей (252 торговых дня)', y=1.02)
plt.tight_layout(); plt.show()

**Вывод 6:** `Close` нестационарна (p≈1) — корректно. Все 9 фичей по ADF дают p≪0.05, т.е. стационарны. Однако rolling-картинка показывает периоды повышенной волатильности (2020 — COVID, 2022 — инфляция); сами уровни средних держатся стабильно. Для baseline это безопасно — можно использовать как есть, без дифференцирования. Если позже захотим использовать ATR/абсолютные return-метрики напрямую — стоит подумать о нормировке скользящей std.

## 7. Стратегия сплита train/val/test и проверка на data leakage

Финансовые ряды нельзя перемешивать (`shuffle=True`) — это сразу даёт лик: модель увидит будущее при обучении и оценке. Нужен строго **временной** сплит.

### Предлагаемый сплит

| Часть | Период | ~Доля |
|---|---|---|
| Train | 2010-01-01 … 2021-12-31 | 12 лет (~80%) |
| Val | 2022-01-01 … 2023-12-31 | 2 года (~13%) |
| Test | 2024-01-01 … 2024-12-31 | 1 год (~7%) |

Train содержит обычные годы + COVID-шок 2020 (полезно для робастности). Val — инфляционный цикл 2022 + AI-ралли 2023 (другой режим, проверим обобщение). Test — последний полный год, исключительно out-of-sample.

Альтернатива для CV: `sklearn.model_selection.TimeSeriesSplit(n_splits=5)` — 5 fold'ов с растущим train-окном, без shuffle. Использовать в `02_baseline` для подбора порога.

In [ ]:
# Визуализация: цена с разделителями train/val/test
df = datasets[PRIMARY]
TRAIN_END = '2021-12-31'
VAL_END = '2023-12-31'

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.index, df['Close'], color='steelblue', linewidth=0.8)
ax.axvspan(df.index.min(), pd.Timestamp(TRAIN_END), alpha=0.10, color='green', label='Train')
ax.axvspan(pd.Timestamp(TRAIN_END), pd.Timestamp(VAL_END), alpha=0.15, color='orange', label='Val')
ax.axvspan(pd.Timestamp(VAL_END), df.index.max(), alpha=0.20, color='red', label='Test')
ax.set_title(f'{PRIMARY}: временной сплит train/val/test')
ax.set_ylabel('Close')
ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

split_summary = pd.DataFrame({
    'Train (≤2021)': [(df.index <= TRAIN_END).sum()],
    'Val (2022-23)': [((df.index > TRAIN_END) & (df.index <= VAL_END)).sum()],
    'Test (2024)':   [(df.index > VAL_END).sum()],
}, index=['n_rows'])
split_summary

### Чек-лист отсутствия data leakage

Анализируем все источники потенциального лика для текущего пайплайна с 9 техническими и 5 макро-фичами:

| Источник лика | Статус | Обоснование |
|---|---|---|
| **Target** использует будущее | ✅ корректно | `Target = (Close.shift(-H) > Close)`, где сейчас `H=5`. Таргет знает будущую цену, но это и есть прогнозируемая величина. Последние `H` строк с NaN таргетом отбрасываются в `build_dataset`. |
| **Технические фичи используют будущее** | ✅ корректно | Все 9 фичей используют только `rolling(window)` назад или `shift(>=0)` назад. RSI/MACD/Bollinger по библиотеке `ta` — тоже только прошлое. Проверено явно ниже. |
| **Макро-фичи используют будущее** | ✅ корректно | `VIX_Change`, `Yield_Spread_Change` и `DXY_Return` считаются через `pct_change()`/`diff()`, а календарь выравнивается через `ffill()`. Это использует только значения `≤ t`. Проверено perturbation-test ниже. |
| **Сплит с shuffle** | ✅ исключаем | В моделях используем `TimeSeriesSplit` или ручной временной разрез — **никогда** `train_test_split(shuffle=True)`. |
| **Scaler/Imputer** на полном датасете | ✅ правило для baseline | Любые трансформации (`StandardScaler`, `RobustScaler`, винзоризация) фитятся **только на train**, потом `.transform()` применяется к val/test. Реализуется через `sklearn.pipeline.Pipeline`. |
| **Утечка через объём/корп. события** | ⚠️ принимаем | yfinance отдаёт уже скорректированные данные на момент скачивания. Если бы мы делали realtime-стратегию — учли бы delay. Для академического upcoming-классификатора игнорируем. |
| **Look-ahead в нормировке фичей по объёму** | ✅ корректно | `Volume_ROC = (V_t - V_{t-1}) / V_{t-1}` — только прошлое. |


In [ ]:
# Проверка причинности фичей: для каждой строки с индексом t — все фичи должны зависеть только от данных <= t.
# Способ проверки: занулим/изменим будущее и убедимся, что фичи на прошлых датах не изменились.
from src.preprocessing import add_features, add_macro_features, load_macro_panel

raw = datasets[PRIMARY][['Open', 'High', 'Low', 'Close', 'Volume']].astype(float).copy()
macro_panel = load_macro_panel(START, END)
split_idx = len(raw) // 2
split_date = raw.index[split_idx]
rng = np.random.default_rng(0)

# Сценарий 1: считаем полный набор фичей на исходном ряду.
feats_full = add_macro_features(add_features(raw), macro_panel)[MODEL_FEATURES]

# Сценарий 2: портим будущие OHLCV и макро-значения после split_date.
raw_perturbed = raw.copy()
raw_noise = 1 + rng.normal(0, 0.5, size=raw_perturbed.iloc[split_idx:, :].shape)
raw_perturbed.iloc[split_idx:, :] = raw_perturbed.iloc[split_idx:, :].to_numpy() * raw_noise

macro_perturbed = macro_panel.copy()
future_macro_mask = macro_perturbed.index >= split_date
macro_noise = 1 + rng.normal(0, 0.5, size=macro_perturbed.loc[future_macro_mask].shape)
macro_perturbed.loc[future_macro_mask] = macro_perturbed.loc[future_macro_mask].to_numpy() * macro_noise
feats_perturbed = add_macro_features(add_features(raw_perturbed), macro_perturbed)[MODEL_FEATURES]

# Если в фичах нет look-ahead — первая половина (≤ split_idx-1) совпадёт точно.
diff = (feats_full.iloc[:split_idx] - feats_perturbed.iloc[:split_idx]).abs().max().max()
print(f'Max abs diff на ПРОШЛОМ участке после изменения БУДУЩЕГО: {diff:.2e}')
print('✅ look-ahead отсутствует' if diff < 1e-10 else '❌ ВНИМАНИЕ: фичи зависят от будущего!')

**Вывод 7:** временной сплит определён, причинность фичей проверена эмпирически. Возмущение будущих OHLCV и макро-рядов не влияет на прошлые значения `MODEL_FEATURES`, значит технические и макро-фичи не используют look-ahead. Все правила анти-лика для моделирования зафиксированы выше.

## 8. Метрика качества и обоснование выбора

### Главная метрика: **Precision** при фиксированном пороге

**Определение:** `Precision = TP / (TP + FP)` — доля правильных «BUY»-сигналов среди всех сделанных сигналов.

**Экономическое обоснование.** Свинг-стратегия открывает позицию по сигналу `prediction = 1`. На дневном таймфрейме структура P&L такова:

- **Истинно положительный (TP):** правильно угадали рост → выигрыш ≈ ожидаемая дневная доходность – комиссия.
- **Ложно положительный (FP):** угадали неверно → проигрыш + комиссия.
- **Истинно/ложно отрицательный (TN/FN):** не открывали позицию → P&L ≈ 0 (упущенная выгода в FN, но без транзакционных издержек).

Поскольку медианная дневная |return| ≈ 0.7%, а round-trip комиссия+slippage у ритейл-брокера ≈ 0.1–0.3%, **цена FP сравнима с выигрышем от TP**. Чтобы стратегия имела положительное матожидание, нужна Precision значимо выше 50% — это и оптимизируем напрямую.

**Реализация:** в `02_baseline` подбираем порог через PR-кривую так, чтобы Precision ≥ 0.55–0.60 при максимально возможном Recall.

### Вспомогательные метрики

| Метрика | Зачем смотрим |
|---|---|
| **ROC-AUC** | Общая разделимость классов, не зависит от порога. Хорошо для сравнения моделей. |
| **PR-AUC (Average Precision)** | Учитывает лёгкий имбаланс лучше ROC-AUC; интегрирует Precision по всем порогам. |
| **Recall** при выбранном пороге | Контролируем, чтобы стратегия не выродилась в «никогда не торгуем» (Precision=1 на одной правильной сделке = бесполезно). |
| **Frequency of trades** (доля `prediction=1`) | Прокси частоты сделок — должна быть достаточной для статистики. |

### Что НЕ используем в качестве главной метрики

- **Accuracy** — нерепрезентативна при имбалансе ≠ 50/50 (на ^GSPC 54% дней растут, тривиальный «всегда BUY» даст Accuracy=0.54, но Precision не выше базы).
- **F1** — равные веса FP и FN, что не соответствует экономике задачи: FP стоит денег, FN — упущенная выгода ≈ 0.
- **MSE/MAE** — мы решаем классификацию, не регрессию.

### Финальный бэктест

После выбора модели и порога — на test-выборке считаем:
- Cumulative return стратегии vs Buy & Hold (^GSPC).
- Sharpe ratio и max drawdown.
- Сравнение Precision на val vs test (стабильность).

## 9. Финальные выводы

**Качество данных.** Чисто: NaN нет, OHLC-инварианты выполнены, дубликатов индекса нет. Макро-колонки присутствуют у всех тикеров, после `build_dataset(..., include_macro=True)` пропусков нет, а значения макро-фичей совпадают на общих датах между тикерами.

**Баланс классов.** Перекос «вверх» умеренный и зависит от выбранного горизонта `H=5`. Метрика — Precision, поэтому имбаланс не критичен, но порог нужно подбирать на validation, а не фиксировать на 0.5.

**Распределения фичей.** `Daily_Return`, `Lag_Return_1`, `Volume_ROC` и часть макро-изменений имеют тяжёлые хвосты. `VIX_Level` и `Yield_Spread` лучше читать как режимные признаки: экстремальный VIX отражает стресс, а отрицательный спред — инверсию кривой. Для линейных моделей стоит использовать `RobustScaler`/винзоризацию; для деревьев можно оставить исходные значения.

**Предсказательная сила (Feature vs Target).** Индивидуальный сигнал каждой фичи слабый, что согласуется с эффективностью рынка. Макро-фичи полезны не как одиночный сигнал, а как контекст: один и тот же VIX/Yield/DXY режим может по-разному менять вероятность роста для `^GSPC`, tech, financials и energy.

**Мультиколлинеарность.** Ожидаемая техническая связка: `Price_to_SMA20` ↔ `Bollinger_pctB` ↔ `MACD_Histogram`. Макро-корреляции стоит интерпретировать отдельно: `VIX_Level`, `Yield_Spread` и `DXY_Return` описывают разные рыночные режимы, но полный VIF важен для LogReg/MLP. Для бустинга оставляем все `MODEL_FEATURES`, для линейных моделей можно исключать наиболее коллинеарные технические признаки.

**Стационарность.** Сырая цена нестационарна. Доходности, изменения и большинство технических фичей должны проходить ADF лучше, чем уровневые макро-переменные. Поэтому `VIX_Level` и `Yield_Spread` используем как regime-фичи, а `VIX_Change`, `Yield_Spread_Change`, `DXY_Return` — как более локальные изменения режима.

**Data leakage.** Проверка причинности теперь покрывает весь `MODEL_FEATURES`: возмущение будущих OHLCV и макро-рядов не должно менять прошлые значения фичей. Это подтверждает, что `rolling`, `shift`, `pct_change`, `diff` и `ffill` не добавляют look-ahead.

**Связь с `03_experiments.ipynb`.** EDA теперь соответствует улучшенному эксперименту `include_macro=True`: модель получает 9 технических + 5 макро-фичей на горизонте `H=5`. Это делает сравнение baseline, `+macro+5d` и pooled training интерпретируемым: добавленные макро-фичи объясняют не локальную динамику свечи, а рыночный режим, в котором технические сигналы работают сильнее или слабее.